# 使用离线引擎查询 VLM

本教程演示如何使用 SGLang 的**离线引擎 API** 查询 VLM。我们将以 Qwen2.5-VL 和 Llama 4 为例进行演示。本节展示三种不同的调用方式：

1. **基本调用**：直接传递图像和文本。
2. **处理器输出**：使用 HuggingFace 处理器进行数据预处理。
3. **预计算嵌入**：预先计算图像特征以提高推理效率。

## 理解三种输入格式

SGLang 支持三种传递视觉数据的方式，每种方式针对不同场景进行了优化：

### 1. **原始图像** - 最简单的方式
- 直接传递 PIL Image、文件路径、URL 或 base64 字符串
- SGLang 自动处理所有预处理
- 最适合：快速原型设计、简单应用

### 2. **处理器输出** - 用于自定义预处理
- 使用 HuggingFace 处理器预处理图像
- 使用 `format: "processor_output"` 传递完整的处理器输出字典
- 最适合：自定义图像变换、与现有流水线集成
- 要求：必须使用 `input_ids` 代替文本提示

### 3. **预计算嵌入** - 获得最佳性能
- 使用视觉编码器预先计算视觉嵌入
- 使用 `format: "precomputed_embedding"` 传递嵌入
- 最适合：对同一图像的重复查询、缓存、高吞吐量服务
- 性能提升：避免冗余的视觉编码器计算（加速 30-50%）

**关键规则**：在单个请求中，所有图像只能使用一种格式。不要混合使用不同格式。

以下示例展示了 Qwen2.5-VL 和 Llama 4 模型的所有三种方式。

## 查询 Qwen2.5-VL 模型

In [ ]:
import nest_asyncio

nest_asyncio.apply()

model_path = "Qwen/Qwen2.5-VL-3B-Instruct"
chat_template = "qwen2-vl"

In [ ]:
from io import BytesIO
import requests
from PIL import Image

from sglang.srt.parser.conversation import chat_templates

image = Image.open(
    BytesIO(
        requests.get(
            "https://github.com/sgl-project/sglang/blob/main/examples/assets/example_image.png?raw=true"
        ).content
    )
)

conv = chat_templates[chat_template].copy()
conv.append_message(conv.roles[0], f"What's shown here: {conv.image_token}?")
conv.append_message(conv.roles[1], "")
conv.image_data = [image]

print("Generated prompt text:")
print(conv.get_prompt())
print(f"\nImage size: {image.size}")
image

### 基本离线引擎 API 调用

In [ ]:
from sglang import Engine

llm = Engine(model_path=model_path, chat_template=chat_template, log_level="warning")

In [ ]:
out = llm.generate(prompt=conv.get_prompt(), image_data=[image])
print("Model response:")
print(out["text"])

### 使用处理器输出调用

使用 HuggingFace 处理器预处理文本和图像，并将 `processor_output` 直接传入 `Engine.generate`。

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(model_path, use_fast=True)
processor_output = processor(
    images=[image], text=conv.get_prompt(), return_tensors="pt"
)

out = llm.generate(
    input_ids=processor_output["input_ids"][0].detach().cpu().tolist(),
    image_data=[dict(processor_output, format="processor_output")],
)
print("Response using processor output:")
print(out["text"])

### 使用预计算嵌入调用

您可以预先计算图像特征以避免重复的视觉编码过程。

In [ ]:
from transformers import AutoProcessor
from transformers import Qwen2_5_VLForConditionalGeneration

processor = AutoProcessor.from_pretrained(model_path, use_fast=True)
vision = (
    Qwen2_5_VLForConditionalGeneration.from_pretrained(model_path).eval().visual.cuda()
)

In [ ]:
processor_output = processor(
    images=[image], text=conv.get_prompt(), return_tensors="pt"
)

input_ids = processor_output["input_ids"][0].detach().cpu().tolist()

precomputed_embeddings = vision(
    processor_output["pixel_values"].cuda(), processor_output["image_grid_thw"].cuda()
)

multi_modal_item = dict(
    processor_output,
    format="precomputed_embedding",
    feature=precomputed_embeddings,
)

out = llm.generate(input_ids=input_ids, image_data=[multi_modal_item])
print("Response using precomputed embeddings:")
print(out["text"])

llm.shutdown()

## 查询 Llama 4 视觉模型

```python
model_path = "meta-llama/Llama-4-Scout-17B-16E-Instruct"
chat_template = "llama-4"

from io import BytesIO
import requests
from PIL import Image

from sglang.srt.parser.conversation import chat_templates

# 下载同一示例图像
image = Image.open(
    BytesIO(
        requests.get(
            "https://github.com/sgl-project/sglang/blob/main/examples/assets/example_image.png?raw=true"
        ).content
    )
)

conv = chat_templates[chat_template].copy()
conv.append_message(conv.roles[0], f"What's shown here: {conv.image_token}?")
conv.append_message(conv.roles[1], "")
conv.image_data = [image]

print("Llama 4 generated prompt text:")
print(conv.get_prompt())
print(f"Image size: {image.size}")

image
```

### Llama 4 基本调用

Llama 4 需要更多计算资源，因此配置了多 GPU 并行（tp_size=4）和更大的上下文长度。

```python
llm = Engine(
    model_path=model_path,
    enable_multimodal=True,
    attention_backend="fa3",
    tp_size=4,
    context_length=65536,
)

out = llm.generate(prompt=conv.get_prompt(), image_data=[image])
print("Llama 4 response:")
print(out["text"])
```

### 使用处理器输出调用

使用 HuggingFace 处理器预处理数据可以减少推理过程中的计算开销。

```python
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(model_path, use_fast=True)
processor_output = processor(
    images=[image], text=conv.get_prompt(), return_tensors="pt"
)

out = llm.generate(
    input_ids=processor_output["input_ids"][0].detach().cpu().tolist(),
    image_data=[dict(processor_output, format="processor_output")],
)
print("Response using processor output:")
print(out)
```

### 使用预计算嵌入调用

```python
from transformers import AutoProcessor
from transformers import Llama4ForConditionalGeneration

processor = AutoProcessor.from_pretrained(model_path, use_fast=True)
model = Llama4ForConditionalGeneration.from_pretrained(
    model_path, torch_dtype="auto"
).eval()

vision = model.vision_model.cuda()
multi_modal_projector = model.multi_modal_projector.cuda()

print(f'Image pixel values shape: {processor_output["pixel_values"].shape}')
input_ids = processor_output["input_ids"][0].detach().cpu().tolist()

# 通过视觉编码器处理图像
image_outputs = vision(
    processor_output["pixel_values"].to("cuda"), 
    aspect_ratio_ids=processor_output["aspect_ratio_ids"].to("cuda"),
    aspect_ratio_mask=processor_output["aspect_ratio_mask"].to("cuda"),
    output_hidden_states=False
)
image_features = image_outputs.last_hidden_state

# 展平图像特征并通过多模态投影器
vision_flat = image_features.view(-1, image_features.size(-1))
precomputed_embeddings = multi_modal_projector(vision_flat)

# 构建预计算嵌入数据项
mm_item = dict(
    processor_output, 
    format="precomputed_embedding", 
    feature=precomputed_embeddings
)

# 使用预计算嵌入进行高效推理
out = llm.generate(input_ids=input_ids, image_data=[mm_item])
print("Llama 4 precomputed embedding response:")
print(out["text"])
```